# TC-WPN — Phase 3B: one configuration per session

**Accelerator: GPU T4. Run this FOUR times, changing `CONFIG` each time.
Always via Save Version → Save & Run All (Commit). Never interactively.**

## What went wrong with Phase 3A

Three separate problems, all of them mine.

**1. The job did not fit in a session.** 20 runs at ~48 min is 16 hours against a
12-hour cap. I wrote that number in the notebook and then still wrote a single
cell that attempts all 20 in one pass. It ran until Kaggle killed it.

**2. The output directory reached roughly 8.6 GB.** `train.py` writes a ~440 MB
`best.pt` per run, and nothing deleted them. Twenty checkpoints in
`/kaggle/working` makes committing extremely slow and makes the notebook's
output listing very heavy to render — the most likely reason the page will not
load now.

**3. Interactive sessions lose `/kaggle/working` when they time out.** If Phase
3A was run interactively rather than committed, the outputs were not saved. See
the recovery section at the bottom before assuming the 12 hours are gone.

## What changed here

- **One configuration per session.** 5 seeds × ~48 min ≈ 4 hours, comfortably
  inside the cap with headroom for a slow T4.
- **Checkpoints are deleted after evaluation**, except for the seed used in
  DeLong pairing and blinded evaluation. Output drops from ~8.6 GB to a few
  hundred MB.
- **Subprocess output goes to a log file**, with only the tail printed. The
  notebook JSON stays small enough to open.
- **Fully resumable.** Anything already evaluated is skipped.

In [ ]:
# ===========================================================================
# SET THIS, THEN COMMIT. Run once per value:
#     "aux_only"  ->  "temporal_aux"  ->  "pcw_aux"  ->  "tcwpn_full"
# ===========================================================================
CONFIG = "aux_only"

SEEDS          = [42, 43, 44, 45, 46]
KEEP_CKPT_SEED = 42      # only this seed's best.pt survives, for DeLong + blinding
K              = 5
STEM           = "psych_mimic4idx"
print("this session trains:", CONFIG, "seeds", SEEDS)

In [ ]:
!rm -rf /kaggle/working/tcwpn_test
!git clone -q https://github.com/dulhara79/tcwpn_test.git /kaggle/working/tcwpn_test
%cd /kaggle/working/tcwpn_test
!pip install -q -r requirements.txt 2>&1 | tail -2

import os
# Quieten the libraries that generate most of the notebook's output volume.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONPATH"] = "src"

import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/test_repo_layout.py", "tests/test_call_arity.py",
                    "-q", "--no-header"],
                   capture_output=True, text=True, env=os.environ)
print(r.stdout[-1500:])
if r.returncode != 0:
    raise SystemExit("Repository layout is broken — fix before spending GPU time.")

In [ ]:
from pathlib import Path
STAGE_A_DS = Path("/kaggle/input/datasets/dulharakaushalya/tc-wpn-stage-a-data")
STAGE_A = next((c for c in (STAGE_A_DS/"data"/"clean", STAGE_A_DS/"clean", STAGE_A_DS)
                if (c/"pkl").exists()), None)
if STAGE_A is None:
    raise SystemExit(f"no pkl/ under {STAGE_A_DS}")

PKL_DIR  = "/kaggle/working/pkl"
PLAN_DIR = str(STAGE_A/"plans")
RESULTS  = "/kaggle/working/results"
LOGS     = "/kaggle/working/logs"
!mkdir -p {PKL_DIR} {LOGS}
!cp {STAGE_A}/pkl/*.pkl {PKL_DIR}/
print("ready")

## Preflight

Supervisor's Stage 2. Everything below is verified before a single GPU minute is
spent, and the cell refuses to continue if any hard check fails. The leakage
rate is read from the certificate rather than assumed — the point of the
certificate is that it gets checked, not filed.

Note the budget rule: the stop threshold is **9 hours**, not 11. A loaded T4 can
run well over 48 min/seed, and being killed at the cap is exactly what cost the
last session.

In [ ]:
# ===========================================================================
# PREFLIGHT — hard gate before any training
# ===========================================================================
import os, json, glob, shutil, math
from pathlib import Path

MIN_FREE_GB   = 25.0    # pkls + checkpoints + logs, with headroom
EST_MIN_PER_RUN = 48
STOP_AFTER_H  = 9.0     # deliberately below the 12 h cap

checks, hard_fail = [], False

def chk(label, ok, detail="", fatal=True):
    global hard_fail
    checks.append((label, ok, detail, fatal))
    if fatal and not ok:
        hard_fail = True
    return ok

# --- repository -----------------------------------------------------------
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pytest",
                    "tests/test_repo_layout.py", "tests/test_call_arity.py",
                    "-q", "--no-header"],
                   capture_output=True, text=True, env=os.environ)
chk("repository tests", r.returncode == 0,
    r.stdout.strip().splitlines()[-1] if r.stdout.strip() else "")

# --- config ---------------------------------------------------------------
cfg_path = f"configs/{CONFIG}.yaml"
chk("configuration file", os.path.exists(cfg_path), cfg_path)

# --- Stage A artefacts ----------------------------------------------------
chk("Stage-A dataset", STAGE_A is not None and Path(STAGE_A).exists(), str(STAGE_A))

pkls = sorted(glob.glob(f"{PKL_DIR}/{STEM}_*.pkl"))
need_pkl = [f"{STEM}_{s}.pkl" for s in ("train", "val", "test")]
have_pkl = {os.path.basename(x) for x in pkls}
chk("PKL files", all(n in have_pkl for n in need_pkl),
    f"{len(pkls)} found; missing {[n for n in need_pkl if n not in have_pkl]}")

need_plan = [f"{STEM}_{s}_k{K}.json" for s in ("train", "val", "test")]
missing_plan = [n for n in need_plan if not os.path.exists(f"{PLAN_DIR}/{n}")]
chk("episode plans", not missing_plan, f"missing {missing_plan}" if missing_plan else "train/val/test present")

# --- leakage certificate --------------------------------------------------
cert_path = f"{PLAN_DIR}/leakage_certificate_{STEM}.json"
leak_detail, leak_ok = "not found", False
if os.path.exists(cert_path):
    cert = json.load(open(cert_path))
    per_k = cert.get("per_k", {}).get(str(K), {})
    rates = {split: blk.get("leakage_rate")
             for split, blk in per_k.items() if isinstance(blk, dict)}
    leak_ok = bool(rates) and all(v == 0 for v in rates.values())
    leak_detail = "  ".join(f"{s}={v:.4%}" for s, v in rates.items()) or "no K entry"
chk("leakage certificate", os.path.exists(cert_path), cert_path)
chk(f"K={K} leakage == 0", leak_ok, leak_detail)

# --- disk -----------------------------------------------------------------
free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
chk("disk space", free_gb >= MIN_FREE_GB, f"{free_gb:.1f} GB free (need {MIN_FREE_GB})")

# --- secrets --------------------------------------------------------------
secret_like = [f for f in glob.glob("**/*", recursive=True)
               if os.path.isfile(f)
               and os.path.basename(f) in {".env", "kaggle.json", "credentials.json"}]
chk("no secret files in repo", not secret_like, str(secret_like), fatal=False)

# --- existing results -----------------------------------------------------
existing = sum(1 for s in SEEDS
               if os.path.exists(f"{RESULTS}/{STEM}/{CONFIG}_k{K}_seed{s}/eval_test.json"))
remaining = len(SEEDS) - existing
est_h = remaining * EST_MIN_PER_RUN / 60
chk("runtime within budget", est_h <= STOP_AFTER_H,
    f"~{est_h:.1f} h for {remaining} run(s); stop threshold {STOP_AFTER_H} h", fatal=False)

# ===========================================================================
print("=" * 60)
print("TC-WPN PHASE 3B PREFLIGHT")
print("=" * 60)
print()
print(f"Configuration: {CONFIG}")
print(f"Seeds:         {','.join(str(s) for s in SEEDS)}")
print(f"K-shot:        {K}")
print()
for label, ok, detail, fatal in checks:
    mark = "OK  " if ok else ("FAIL" if fatal else "WARN")
    print(f"  {label:<26} {mark}   {detail}")
print()
print(f"Existing runs:       {existing}")
print(f"Runs remaining:      {remaining}")
print(f"Estimated runtime:   ~{est_h:.1f} h")
print()
print("Checkpoint policy:")
for s in SEEDS:
    print(f"  seed {s} -> {'KEEP' if s == KEEP_CKPT_SEED else 'DELETE after evaluation'}")
print()
if hard_fail:
    print("STATUS: BLOCKED — fix the FAIL rows above before starting")
    print("=" * 60)
    raise SystemExit("preflight failed")
if est_h > STOP_AFTER_H:
    print(f"STATUS: SAFE TO START, BUT split the seeds across two commits")
    print(f"        (~{est_h:.1f} h exceeds the {STOP_AFTER_H} h stop threshold)")
else:
    print("STATUS: SAFE TO START")
print("=" * 60)

## Recover anything that already exists

Add every previous result dataset as an input. Predictions files are preferred
over checkpoints: DeLong needs the score vectors, not the model.

In [ ]:
import glob, shutil, os
import pandas as pd

# Four distinct states, per the supervisor's point 2. The previous version
# conflated "predictions recovered" with "ready", so a run holding only a
# predictions CSV was skipped by the training loop AND had no eval_test.json,
# leaving it invisible to aggregation.
#
#   complete       eval_test.json + predictions_test.csv   -> nothing to do
#   predictions    predictions_test.csv only               -> usable for DeLong,
#                                                             NOT for mean+/-SD
#   checkpoint     best.pt only                            -> re-evaluate (cheap)
#   missing        nothing                                 -> train (expensive)

def classify(cfg, seed):
    name = f"{cfg}_k{K}_seed{seed}"
    dst  = f"{RESULTS}/{STEM}/{name}"
    has_eval = os.path.exists(f"{dst}/eval_test.json")
    has_pred = os.path.exists(f"{dst}/predictions_test.csv")
    has_ckpt = os.path.exists(f"{dst}/best.pt")

    if not (has_eval or has_pred or has_ckpt):
        for target in ("eval_test.json", "predictions_test.csv", "best.pt"):
            hits = glob.glob(f"/kaggle/input/**/{name}/{target}", recursive=True)
            if hits:
                src = os.path.dirname(sorted(hits)[0])
                os.makedirs(dst, exist_ok=True)
                for f in os.listdir(src):
                    q = os.path.join(src, f)
                    if os.path.isfile(q):
                        shutil.copy2(q, dst)
                break
        has_eval = os.path.exists(f"{dst}/eval_test.json")
        has_pred = os.path.exists(f"{dst}/predictions_test.csv")
        has_ckpt = os.path.exists(f"{dst}/best.pt")

    if has_eval and has_pred:
        return "complete"
    if has_ckpt:
        return "checkpoint"      # re-evaluate; regenerates both artefacts
    if has_pred:
        return "predictions"     # DeLong-usable only
    return "missing"

status = {s: classify(CONFIG, s) for s in SEEDS}
print(pd.Series(status, name="state").to_string())
print()

counts = {k: sum(1 for v in status.values() if v == k) for k in
          ("complete", "checkpoint", "predictions", "missing")}
print(f"complete    : {counts['complete']}   nothing to do")
print(f"checkpoint  : {counts['checkpoint']}   re-evaluate (~2 min each)")
print(f"predictions : {counts['predictions']}   DeLong-usable, but NO metrics -> retrain for mean+/-SD")
print(f"missing     : {counts['missing']}   train (~48 min each)")

to_train = counts["missing"] + counts["predictions"]
print(f"\nGPU work this session: ~{to_train*48/60:.1f} h")
if counts["predictions"]:
    print("\nNOTE: a predictions-only run contributes to the paired DeLong test but")
    print("      cannot contribute to the mean +/- SD table, because eval_test.json")
    print("      holds the metrics. Those seeds are retrained below.")

## Train, evaluate, then free the disk

Each run's stdout goes to `/kaggle/working/logs/`. Only the last 25 lines print,
which is enough to see the loss trajectory and the locked threshold without
inflating the notebook. The full log is committed with the output if you need it.

In [ ]:
import subprocess, os, time

def run(cmd, logfile, tail=25):
    t0 = time.time()
    with open(logfile, "w") as fh:
        p = subprocess.run(cmd, stdout=fh, stderr=subprocess.STDOUT, env=os.environ)
    mins = (time.time() - t0) / 60
    lines = open(logfile).read().splitlines()
    print("\n".join(lines[-tail:]))
    print(f"[exit {p.returncode} | {mins:.1f} min | full log: {logfile}]")
    return p.returncode

session_start = time.time()

for seed in SEEDS:
    elapsed_h = (time.time() - session_start) / 3600
    if elapsed_h > STOP_AFTER_H:
        print(f"\nSTOPPING at {elapsed_h:.1f} h to stay clear of the session cap.")
        print(f"Remaining seeds: {[s for s in SEEDS if s >= seed]}")
        print("Commit now, then rerun this notebook — it resumes from here.")
        break

    name    = f"{CONFIG}_k{K}_seed{seed}"
    run_dir = f"{RESULTS}/{STEM}/{name}"
    state   = status[seed]

    if state == "complete":
        print(f"skip {name}: already complete"); continue

    # 'predictions' has no checkpoint and no metrics -> must retrain.
    if state == "missing" or state == "predictions":
        if state == "predictions":
            print(f"{name}: predictions present but no metrics/checkpoint -> retraining")
        print("=" * 70); print(f"TRAIN {name}"); print("=" * 70)
        rc = run(["python", "-m", "scripts.train",
                  "--config", f"configs/{CONFIG}.yaml",
                  "--k", str(K), "--seed", str(seed), "--stem", STEM,
                  "--pkl-dir", PKL_DIR, "--plan-dir", PLAN_DIR,
                  "--results", RESULTS],
                 f"{LOGS}/train_{name}.log")
        if rc != 0:
            print(f"TRAIN FAILED for {name}; skipping evaluation"); continue

    if not os.path.exists(f"{run_dir}/eval_test.json"):
        print("-" * 70); print(f"EVALUATE {name}"); print("-" * 70)
        run(["python", "-m", "scripts.evaluate", "--run", run_dir,
             "--split", "test", "--pkl-dir", PKL_DIR, "--plan-dir", PLAN_DIR,
             "--bootstrap", "2000"],
            f"{LOGS}/eval_{name}.log")

    status[seed] = "complete" if os.path.exists(f"{run_dir}/eval_test.json") else status[seed]

    # Free ~440 MB — but only for seeds that are not needed downstream.
    if seed != KEEP_CKPT_SEED and os.path.exists(f"{run_dir}/best.pt"):
        if os.path.exists(f"{run_dir}/eval_test.json") and \
           os.path.exists(f"{run_dir}/predictions_test.csv"):
            os.remove(f"{run_dir}/best.pt")
            print(f"[removed best.pt for seed {seed}: metrics and predictions are saved]")
        else:
            print(f"[KEEPING best.pt for seed {seed}: artefacts incomplete]")

!du -sh /kaggle/working/results /kaggle/working/pkl /kaggle/working/logs

## Blinded evaluation — only in the session that keeps the checkpoint

In [ ]:
# Seed 42's checkpoint is retained until ALL of the following exist:
#   eval_test.json, predictions_test.csv, eval_test_blind-anxiety.json,
#   eval_test_blind-anx_meds.json
# Supervisor's point 3: do not delete it before the blinded arms are done.
run_dir = f"{RESULTS}/{STEM}/{CONFIG}_k{K}_seed{KEEP_CKPT_SEED}"

if os.path.exists(f"{run_dir}/best.pt"):
    for level in ["anxiety", "anx_meds"]:
        if os.path.exists(f"{run_dir}/eval_test_blind-{level}.json"):
            print(f"skip blinded {level}: already done"); continue
        blind_pkl = f"{PKL_DIR}/{STEM}_test_blind-{level}.pkl"
        if not os.path.exists(blind_pkl):
            print(f"MISSING {blind_pkl} — build it in Stage A with "
                  f"tokenize_cohort --blind {level}"); continue
        print(f"blinded evaluation: {level}")
        run(["python", "-m", "scripts.evaluate", "--run", run_dir,
             "--split", "test", "--blind", level,
             "--pkl-dir", PKL_DIR, "--plan-dir", PLAN_DIR, "--bootstrap", "2000"],
            f"{LOGS}/eval_blind_{level}_{CONFIG}.log")

    required = ["eval_test.json", "predictions_test.csv",
                "eval_test_blind-anxiety.json", "eval_test_blind-anx_meds.json"]
    have = [f for f in required if os.path.exists(f"{run_dir}/{f}")]
    print()
    print(f"seed {KEEP_CKPT_SEED} artefacts: {len(have)}/{len(required)} present")
    for f in required:
        print(f"   {'OK ' if f in have else '-- '} {f}")
    print("\nCheckpoint retained (needed for DeLong pairing and any re-run)."
          if len(have) < len(required) else
          "\nAll artefacts present. Checkpoint still retained for DeLong pairing.")
else:
    print("no checkpoint retained in this session — nothing to blind")

In [ ]:
# This session's numbers. The cross-config aggregation happens in Phase 3C,
# after all four sessions have been committed.
import json, glob
rows = []
for f in sorted(glob.glob(f"{RESULTS}/{STEM}/{CONFIG}_k{K}_seed*/eval_test.json")):
    m = json.load(open(f))["metrics"]
    rows.append({"run": os.path.basename(os.path.dirname(f)),
                 "AUROC": round(m["auroc"], 4),
                 "CI_low": round(m["auroc_ci_lower"], 4),
                 "CI_high": round(m["auroc_ci_upper"], 4),
                 "PR_AUC": round(m["pr_auc"], 4),
                 "F1": round(m["f1_positive"], 4),
                 "Sens": round(m["sensitivity"], 4),
                 "Spec": round(m["specificity"], 4)})
if rows:
    df = pd.DataFrame(rows).set_index("run")
    print(df.to_string())
    print(f"\nAUROC mean {df['AUROC'].mean():.4f}  SD {df['AUROC'].std(ddof=1):.4f}")
    df.to_csv(f"/kaggle/working/{CONFIG}_seed_results.csv")
else:
    print("no completed runs in this session")
print("\nNow: Save Version -> Save & Run All (Commit).")
print("Then change CONFIG and repeat. After all four, run Phase 3C to aggregate.")

## If the Phase 3A page still will not open

The notebook is probably too large to render, not corrupted. Options in order:

1. **Kaggle API from your own machine** — this does not load the page at all:
   ```
   pip install kaggle
   kaggle kernels output dulharakaushalya/tc-wpn-phase-3-five-seeds-the-completed-delon -p ./phase3a_output
   ```
   If Phase 3A was committed, every file it wrote comes down, including any
   `eval_test.json` and `predictions_test.csv` that completed before the cutoff.
   Those are worth recovering — each one is a run you do not have to repeat.

2. **The notebook's Output tab** rather than the editor. It renders the file
   listing without the cell outputs.

3. **Copy & Edit** to fork it. The fork loads a fresh editor; you can then clear
   all outputs and save.

If Phase 3A ran **interactively** rather than as a commit, `/kaggle/working` was
discarded when the session was killed and there is nothing to recover. That is
the case worth checking first, because it determines whether you restart from
zero or from partial results.

Either way, upload whatever you recover as a Kaggle dataset and add it as an
input here — the recovery cell will find it.